# K리그 최종 패스 좌표 예측 AI 모델 - Colab GPU 학습

이 노트북은 Google Colab의 GPU를 사용하여 모델을 학습합니다.

## 사용 방법
1. **런타임 → 런타임 유형 변경 → GPU 선택** (필수!)
2. 아래 셀들을 순서대로 실행
3. 학습 완료 후 결과 파일 다운로드


In [ ]:
# 1. GPU 설정 확인
import torch
print("=" * 50)
print("GPU 설정 확인")
print("=" * 50)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"✓ GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠ GPU를 사용할 수 없습니다!")
    print("런타임 → 런타임 유형 변경 → GPU 선택 후 다시 실행하세요.")


In [ ]:
# 2. 필수 패키지 설치
print("=" * 50)
print("패키지 설치 중...")
print("=" * 50)
%pip install -q torch numpy pandas scikit-learn tqdm
print("✓ 패키지 설치 완료!")


## 📁 데이터 업로드 방법

**옵션 1: Google Drive 사용 (대용량 데이터 권장)**
- Google Drive에 프로젝트 폴더 업로드
- 아래 셀에서 Drive 마운트

**옵션 2: 직접 업로드**
- Colab 파일 메뉴에서 직접 업로드
- `train.csv`, `test.csv`, `test/` 폴더 등

**옵션 3: GitHub에서 클론**
- GitHub에 프로젝트 업로드 후 아래 셀에서 클론


In [ ]:
# 3-1. 옵션: Google Drive 마운트 (선택사항)
# Google Drive에 프로젝트를 업로드한 경우에만 사용
from google.colab import drive
drive.mount('/content/drive')

# 프로젝트 경로 설정 (본인의 Drive 경로로 수정)
# 예: PROJECT_PATH = "/content/drive/MyDrive/DACON-KLeague-AI-Competition"
PROJECT_PATH = "/content/drive/MyDrive/DACON-KLeague-AI-Competition"  # 수정 필요!

import os
if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    print(f"✓ 작업 디렉토리 변경: {os.getcwd()}")
else:
    print(f"⚠ 경로를 찾을 수 없습니다: {PROJECT_PATH}")
    print("본인의 Google Drive 경로로 수정하세요.")


In [ ]:
# 3-2. 옵션: GitHub에서 클론 (선택사항)
# GitHub에 프로젝트를 업로드한 경우에만 사용
# !git clone https://github.com/[your-username]/DACON-KLeague-AI-Competition.git
# %cd DACON-KLeague-AI-Competition
# print(f"✓ 작업 디렉토리: {os.getcwd()}")


In [ ]:
# 4. 필수 파일 확인
import os

print("=" * 50)
print("필수 파일 확인")
print("=" * 50)

required_files = [
    'config.py',
    'data_preprocessing.py', 
    'model.py',
    'train.py',
    'inference.py',
    'train.csv',
    'test.csv',
    'sample_submission.csv'
]

missing_files = []
for file in required_files:
    if os.path.exists(file):
        print(f"✓ {file}")
    else:
        print(f"✗ {file} (누락)")
        missing_files.append(file)

if missing_files:
    print(f"\n⚠ 누락된 파일이 있습니다: {len(missing_files)}개")
    print("다음 방법으로 파일을 업로드하세요:")
    print("1. Colab 파일 메뉴(왼쪽 사이드바)에서 직접 업로드")
    print("2. Google Drive에 업로드 후 마운트")
    print("3. GitHub에서 클론")
else:
    print(f"\n✓ 모든 필수 파일 확인 완료!")
    
# test 폴더 확인
if os.path.exists('test'):
    test_files = sum([len(files) for r, d, files in os.walk('test')])
    print(f"✓ test 폴더: {test_files}개 파일")
else:
    print("✗ test 폴더 없음")


In [ ]:
# 5. Colab 환경 설정 (경로 자동 조정)
import os

# Colab인지 확인
try:
    import google.colab
    IN_COLAB = True
    print("✓ Google Colab 환경 감지")
    
    # Colab에서는 /content가 루트
    if not os.path.exists('train.csv'):
        # 현재 디렉토리 확인
        print(f"현재 디렉토리: {os.getcwd()}")
        print("⚠ train.csv를 찾을 수 없습니다.")
        print("파일을 업로드하거나 경로를 확인하세요.")
    else:
        # config.py의 DATA_DIR을 현재 디렉토리로 설정
        os.environ['COLAB_DATA_DIR'] = os.getcwd()
        print(f"✓ 데이터 디렉토리 설정: {os.getcwd()}")
        
except ImportError:
    IN_COLAB = False
    print("로컬 환경입니다.")


## 🚀 모델 학습

아래 셀을 실행하여 모델을 학습합니다.
GPU를 사용하면 훨씬 빠르게 학습됩니다!


In [ ]:
# 6. 모델 학습 실행
print("=" * 50)
print("모델 학습 시작")
print("=" * 50)
print("GPU를 사용하면 훨씬 빠릅니다!")
print("학습 시간: 약 1-2시간 (Early stopping 고려)")
print()

!python train.py


## 📊 학습 결과 확인


In [ ]:
# 7. 학습된 모델 확인
import os

model_path = 'models/best_model.pth'
if os.path.exists(model_path):
    file_size = os.path.getsize(model_path) / (1024 * 1024)  # MB
    print(f"✓ 모델 파일 생성 완료: {model_path}")
    print(f"  파일 크기: {file_size:.2f} MB")
else:
    print(f"⚠ 모델 파일을 찾을 수 없습니다: {model_path}")


## 🔮 추론 실행

학습이 완료되면 아래 셀을 실행하여 테스트 데이터에 대한 예측을 수행합니다.


In [ ]:
# 8. 추론 실행
print("=" * 50)
print("추론 시작")
print("=" * 50)

!python inference.py


## 📥 결과 다운로드

추론이 완료되면 제출 파일을 다운로드합니다.


In [ ]:
# 9. 결과 파일 다운로드
from google.colab import files
import os

submission_file = 'submission.csv'
if os.path.exists(submission_file):
    print(f"✓ {submission_file} 다운로드 중...")
    files.download(submission_file)
    print("✓ 다운로드 완료!")
else:
    print(f"⚠ 파일을 찾을 수 없습니다: {submission_file}")

# 모델 파일도 다운로드 (선택사항)
model_file = 'models/best_model.pth'
if os.path.exists(model_file):
    print(f"\n모델 파일도 다운로드하시겠습니까? (용량이 큽니다)")
    # 주석 해제하여 다운로드
    # files.download(model_file)


## ✅ 완료!

제출 파일(`submission.csv`)을 다운로드하여 대회에 제출하세요!

### 다음 단계
1. 다운로드한 `submission.csv` 파일 확인
2. 대회 플랫폼에 제출
3. 점수 확인 및 개선

### 참고
- 학습된 모델은 `models/best_model.pth`에 저장됩니다
- Google Drive에 저장하려면 Drive 마운트 후 복사하세요
